# **RChilli inference and evaluation process**

This section contains information about the inference process using the RChilli service.

## **First evaluation**

RChilli outputs a structured JSON containing the extracted information from the resume. The output includes many fields, one of which is called `DetailResume`. This field functions as a parsed version of the entire resume content in the order it was detected—it is not segmented or structured, and its output is more similar to that of an OCR system, but without bounding boxes. Therefore, to evaluate reading order and text extraction metrics, we will use NDLD and BLEU, as well as WER/CER, respectively.

In [1]:
import os
import base64
import re
import json
import pandas as pd
import requests
from dotenv import load_dotenv

In [2]:
load_dotenv()
rchilli_user_id = os.getenv("RCHILLI_USER_KEY")

In [8]:
API_URL = "https://rest.rchilli.com/RChilliParser/Rchilli/parseResumeBinary"
USER_KEY = "rchilli_user_id"
SUBUSER_ID = "someone"  
VERSION = "8.0.0"           
INPUT_DIR = "missing_rchilli"
OUTPUT_DIR = "/Users/elizabethgranda/Documents/resume_parsing/factored_benchmark/resume-parsing-exploration/RChilli_analysis/rchilli_output"

In [2]:
def parse_resume(file_path: str):
    with open(file_path, "rb") as f:
        file_bytes = f.read()
    base64_str = base64.b64encode(file_bytes).decode("utf-8")

    payload = {
        "filedata": base64_str,
        "filename": os.path.basename(file_path),
        "userkey": USER_KEY,
        "version": VERSION,
        "subuserid": SUBUSER_ID
    }

    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json"
    }

    response = requests.post(API_URL, json=payload, headers=headers)
    response.raise_for_status()
    return response.json()


in here we make the inference over the data: rchilli only accepts as an input read-files: pdf, word, etc.

In [7]:
for filename in os.listdir(INPUT_DIR):
    if filename.lower().endswith(".pdf"):
        full_path = os.path.join(INPUT_DIR, filename)
        print(f"Parsing {filename} ...")

        try:
            result_json = parse_resume(full_path)
            output_file = os.path.join(OUTPUT_DIR, f"{os.path.splitext(filename)[0]}.json")
            with open(output_file, "w", encoding="utf-8") as out_f:
                json.dump(result_json, out_f, indent=2, ensure_ascii=False)

            print(f"Saved parsed output to {output_file}")
        except Exception as e:
            print(f"Error parsing {filename}: {e}")

print("Done.")

Parsing german_andres.pdf ...
Saved parsed output to /Users/elizabethgranda/Documents/resume_parsing/factored_benchmark/resume-parsing-exploration/RChilli_analysis/rchilli_output/german_andres.json
Parsing Alejandro'sResume.pdf ...
Saved parsed output to /Users/elizabethgranda/Documents/resume_parsing/factored_benchmark/resume-parsing-exploration/RChilli_analysis/rchilli_output/Alejandro'sResume.json
Parsing AlexisMetauteIngles-2.pdf ...
Saved parsed output to /Users/elizabethgranda/Documents/resume_parsing/factored_benchmark/resume-parsing-exploration/RChilli_analysis/rchilli_output/AlexisMetauteIngles-2.json
Parsing AbdullahAshfaq-JuniorDataScientist.pdf ...
Saved parsed output to /Users/elizabethgranda/Documents/resume_parsing/factored_benchmark/resume-parsing-exploration/RChilli_analysis/rchilli_output/AbdullahAshfaq-JuniorDataScientist.json
Parsing AMPADUHYACINTHKWADWO8.pdf ...
Saved parsed output to /Users/elizabethgranda/Documents/resume_parsing/factored_benchmark/resume-parsing

In [9]:
def extract_detail_resume_from_dir(json_dir):
    rows = []
    for fname in os.listdir(json_dir):
        if not fname.lower().endswith(".json"):
            continue

        file_path = os.path.join(json_dir, fname)

        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        detail_resume = (
            data
            .get("ResumeParserData", {})
            .get("DetailResume", ""))

        rows.append({
            "filename": os.path.splitext(fname)[0],
            "original_text": detail_resume
        })

    return pd.DataFrame(rows)

Now we can make the evaluation

In [10]:
def clean_text(df: pd.DataFrame) -> pd.DataFrame:
    df_copy = df.copy()
    df_copy['formatted_text'] = df_copy['original_text'].str.replace(r'[\n\t\r]', ' ', regex=True)
    return df_copy

In [11]:
def add_words_column(df: pd.DataFrame) -> pd.DataFrame:
    def smart_tokenize(text):
        if pd.isna(text):
            return []
        # attach opening punctuation to the right word
        text = re.sub(r'([(\[\{])\s+', r'\1', text)
        text = re.sub(r'\s+([.,;:!?|\]\)}])', r'\1', text)
        # attach hyphens to the right word, but only if surrounded by spaces
        text = re.sub(r'\s+(-)\s+', r'\1', text)

        text = re.sub(r'\s+', ' ', text).strip()
        return text.split(' ')
    
    df_copy = df.copy()
    df_copy['words'] = df_copy['formatted_text'].apply(smart_tokenize)
    return df_copy

In [12]:
results = clean_text(extract_detail_resume_from_dir(OUTPUT_DIR))
results = add_words_column(results)
results.head()

,filename,original_text,formatted_text,words
0,Rodrigo_Fay_Resume_2021,RODRIGO FAY VERGARA\nMACHINE LEARNING ENGINEER...,RODRIGO FAY VERGARA MACHINE LEARNING ENGINEER ...,"[RODRIGO, FAY, VERGARA, MACHINE, LEARNING, ENG..."
1,AMPADUHYACINTHKWADWO8,AMPADU HYACINTH KWADWO \n\nPERSONAL PROFILE\n...,AMPADU HYACINTH KWADWO PERSONAL PROFILE I ...,"[AMPADU, HYACINTH, KWADWO, PERSONAL, PROFILE, ..."
2,Resume_Louis_Renaux,Louis Renaux\n\nML Scientist specialized in Co...,Louis Renaux ML Scientist specialized in Comp...,"[Louis, Renaux, ML, Scientist, specialized, in..."
3,CVGustavoNunezOrtizENG,GUSTAVO ANDRES\nNUNEZ-ORTIZ\nElectronic Engine...,GUSTAVO ANDRES NUNEZ-ORTIZ Electronic Engineer...,"[GUSTAVO, ANDRES, NUNEZ-ORTIZ, Electronic, Eng..."
4,DorianMoreno_CVG,"Dorian R Moreno\n ( +57 ) 319 2744528\t,dorian...","Dorian R Moreno ( +57 ) 319 2744528 ,dorian.r...","[Dorian, R, Moreno, (+57), 319, 2744528,dorian..."


In [14]:
import pandas as pd
from utils.helpers import OrganizeLSOutput, FormateLSOutputReadingOrder
from utils.ocr_metrics import compute_metrics_wer_cer
from utils.reading_order_metrics import compute_ndld_dataframe
from utils.ocr_metrics import normalize_text
from utils.reading_order_metrics import prepare_refs_hyps, compute_bleu
from utils.ocr_output_norma import BaseJsonNormalization

In [13]:
from typing import Dict, Any
class ImageDimFetcher(BaseJsonNormalization):
    def process_single_file(self, json_file_path: str) -> Dict[str, Any]:
        return {}

In [15]:
ground_truth_data = pd.read_csv("/Users/elizabethgranda/Documents/resume_parsing/factored_benchmark/resume-parsing-exploration/last_annotations_309.csv")

ground_truth_data = FormateLSOutputReadingOrder(ground_truth_data).process()

ground_truth_data = OrganizeLSOutput(
    ground_truth_data,
    transcription_column="ro_transcription",
    rename={"words": "words_ro"},
    images_directory="/Users/elizabethgranda/Documents/resume_parsing/factored_benchmark/resume-parsing-exploration/factored_resumes_images"
).process_all()

File: VedantSaraswat_page_001.png → bio has 83 elements, reading_order has 84
File: mhannani_page_001.png → bio has 87 elements, reading_order has 88


In [16]:
ground_truth_data= ground_truth_data.drop(columns=["secbox", "bio"])

In [17]:
# Update the filename values in ground_truth_data
ground_truth_data.loc[ground_truth_data['filename'] == 'CV_English_page_001', 'filename'] = 'CarlosAndresSanchezAlzate_page_001'
ground_truth_data.loc[ground_truth_data['filename'] == 'CV_English_page_002', 'filename'] = 'CarlosAndresSanchezAlzate_page_002'

As mentioned in previous notebooks, our ground-truth data consists of images of CVs, meaning we do not have "fully annotated resumes per sé", but rather annotated images that together make up each resume. Thats why, in this next phase, **we join the images that belong to the same resume in order to unify them and enable a proper evaluation of the service.**

In [18]:
def unify_resumes(df, id_column):

    df = df.copy()
    df['resume_id'] = df[id_column].str.replace(r'_page_\d+$', '', regex=True)
    df['page_num'] = df[id_column].str.extract(r'_page_(\d+)$').astype(int)
    df.sort_values(['resume_id', 'page_num'], inplace=True)

    unified_rows = []

    for resume_id, group in df.groupby('resume_id'):
        group_sorted = group.sort_values('page_num')
        base_row = group_sorted.iloc[0].drop(labels=['resume_id', 'page_num']).copy()
        base_row[id_column] = resume_id

        for _, row in group_sorted.iloc[1:].drop(columns=['resume_id', 'page_num']).iterrows():
            for col in base_row.index:
                if col == id_column:
                    continue

                val1 = base_row[col]
                val2 = row[col]

                if isinstance(val1, list) and isinstance(val2, list):
                    base_row[col] = val1 + val2
                elif isinstance(val1, list):
                    base_row[col] = val1 + [val2]
                elif isinstance(val2, list):
                    base_row[col] = [val1] + val2
                else:
                    base_row[col] = [val1, val2]

        unified_rows.append(base_row)

    unified_df = pd.DataFrame(unified_rows)
    return unified_df

In [19]:
unified_dataframe = unify_resumes(ground_truth_data, "filename")

In [20]:
results_bow = {}
results_standard = {}

models = {
    'rchilli': results,
}

for model_name, predictions_df in models.items():
    standard_results_df = compute_metrics_wer_cer(predictions_df, unified_dataframe, ground_truth_words_col='words_ro')
    results_standard[model_name] = standard_results_df

standard_summary_data = []
for model_name, results_df in results_standard.items():
    mean_wer = results_df['wer_score'].mean()
    mean_cer = results_df['cer_score'].mean()
    standard_summary_data.append({
        'Model': model_name,
        'Mean WER': mean_wer,
        'Mean CER': mean_cer
    })

standard_summary_table = pd.DataFrame(standard_summary_data)

ground-truth was not found: cv_mateo_campo_en
ground-truth was not found: Resume 2
ground-truth was not found: Resume (7)


In [21]:
standard_summary_table

,Model,Mean WER,Mean CER
0,rchilli,0.12931,0.112416


In [23]:
print("Computing WORD level NDLD...")
scores_word = compute_ndld_dataframe(
    unified_dataframe,
    results,
    key="filename",
    gt_col="words_ro",
    pred_col="words",
    level="word",
    normalize_fn=lambda t: normalize_text(
        t, lowercase=True, remove_punct=True,
        remove_diacritics=True, dehyphenate=True
    )
)

print("Computing CHARACTER level NDLD...")
scores_char = compute_ndld_dataframe(
    unified_dataframe,
    results,
    key="filename",
    gt_col="words_ro",
    pred_col="words",
    level="character",
    normalize_fn=lambda t: normalize_text(
        t, lowercase=True, remove_punct=False,
        remove_diacritics=False, dehyphenate=True
    )
)

summary_df = pd.DataFrame([
    {
        "Level": "Word",
        "Mean_Similarity": scores_word["similarity"].mean(),
        "Mean_NDLD": scores_word["ndld"].mean(),
    },
    {
        "Level": "Character",
        "Mean_Similarity": scores_char["similarity"].mean(),
        "Mean_NDLD": scores_char["ndld"].mean(),
    },
])

Computing WORD level NDLD...
Computing CHARACTER level NDLD...


In [27]:
summary_df

,Level,Mean_Similarity,Mean_NDLD
0,Word,0.829979,0.170021
1,Character,0.885568,0.114432


In [24]:
print("Scoring BLEU...")

hyps, refs = prepare_refs_hyps(
    ensayo_ro=unified_dataframe,
    pred_df=results,
    pred_col="words",
    gt_filename_col="filename",
    gt_text_col="words_ro",
)

metrics = compute_bleu(hyps, refs)

page_bleu_df = pd.DataFrame([{
    "Pages scored": int(metrics["pages_scored"]),
    "Avg Page BLEU": metrics["avg_page_bleu"],
}])

corpus_bleu_df = pd.DataFrame([{
    "Corpus BLEU": metrics["corpus_bleu"],
    "P1": metrics["p1"],
    "P2": metrics["p2"],
    "P3": metrics["p3"],
    "P4": metrics["p4"],
    "Brevity Penalty": metrics["brevity_penalty"],
}])

Scoring BLEU...


In [25]:
page_bleu_df

,Pages scored,Avg Page BLEU
0,112,87.663473


In [26]:
corpus_bleu_df

,Corpus BLEU,P1,P2,P3,P4,Brevity Penalty
0,90.460017,97.342193,92.407402,88.126725,84.471682,1.0


### **Second evaluation**

Rchilli is a resume parser, so the most valuable information it should provide is each section of the resume correctly parsed, in the proper order, and with the correct content. What is evaluated in this section is whether the `Segregated` information within the JSON output is truly separated and structured correctly according to each section. 

However RChilli output is really noisy and contains information we actually don't need and segregations inside segregations that are not important for our study, that's why we need to "recosntruct" the jsons outputs with the sections we want to evaluate, which are the same sections we've been using for evaluating the rest of the VLMs: `personal_info`, `education`, `skills`, `employment_information`, `employment_description`.

So this code below creates that structure and saves the new restructured json files in the given output directory, then **we make the evaluation using the `vlm_evaluation.ipynb` notebook** where we give this structured jsons directory along with the ground-truth structured jsons and make the comparison of BERTScore metric.

In [9]:
import os
import re
import json
from typing import Dict, Any, List


def safe_get(dct, path, default=""):
    for key in path:
        if isinstance(dct, dict):
            dct = dct.get(key)
        else:
            return default
    return dct if dct not in (None, []) else default


def extract_personal_info(rpd):
    email = rpd.get("Email", [{}])
    phone = rpd.get("PhoneNumber", [{}])
    websites = rpd.get("WebSite", [])
    address = rpd.get("Address", [{}])

    linkedin = ""
    github = ""

    for site in websites:
        if site.get("Type", "").lower() == "linkedin":
            linkedin = site.get("Url", "")
        elif site.get("Type", "").lower() == "github":
            github = site.get("Url", "")

    return {
        "full_name": safe_get(rpd, ["Name", "FullName"]),
        "email": email[0].get("EmailAddress", "") if email else "",
        "phone": phone[0].get("OriginalNumber", "") if phone else "",
        "location": address[0].get("FormattedAddress", "") if address else "",
        "github": github,
        "linkedin": linkedin,
    }


def extract_education(rpd):
    education = []

    for edu in rpd.get("SegregatedQualification", []):
        education.append({
            "institution": safe_get(edu, ["Institution", "Name"]),
            "institution_location": safe_get(edu, ["Institution", "Location", "City", "Country"]),
            "degree": safe_get(edu, ["Degree", "DegreeName"]),
            "sub_institution": safe_get(edu, ["SubInstitution", "Name"]),
            "specialization": ", ".join(
                safe_get(edu, ["Degree", "Specialization"], [])
            ) if isinstance(safe_get(edu, ["Degree", "Specialization"]), list) else "",
            "start_date": edu.get("StartDate", ""),
            "end_date": edu.get("EndDate", ""),
        })

    certification_text = rpd.get("Certification", "")
    if certification_text:
        education.append({"certification": certification_text})

    return education


def extract_skills(rpd):
    raw_skills = rpd.get("SkillBlock", "")
    languages = rpd.get("LanguageKnown", [])

    skills = []

    if raw_skills:
        cleaned = (
            raw_skills
            .replace("\t", " ")
            .replace("\r", " ")
            .replace("\n", " ")
        )
        skills.extend([
            s.strip()
            for s in re.split(r"\s{2,}", cleaned)
            if s.strip() and not re.match(r"^\d+%$", s.strip())
        ])

    for lang in languages:
        language_name = lang.get("Language", "")
        if language_name:
            skills.append(language_name)

    seen = set()
    deduped_skills = []
    for s in skills:
        if s not in seen:
            seen.add(s)
            deduped_skills.append(s)

    return deduped_skills


def extract_experience(rpd: Dict[str, Any]) -> List[Dict[str, str]]:
    experience = []

    for exp in rpd.get("SegregatedExperience", []):
        location = exp.get("Location", {})
        city = location.get("City", "")
        state = location.get("State", "")

        experience.append({
            "company": safe_get(exp, ["Employer", "EmployerName"]),
            "role": safe_get(exp, ["JobProfile", "Title"]),
            "location": city or state,
            "start_date": exp.get("StartDate", ""),
            "end_date": exp.get("EndDate", ""),
            "employment_description": exp.get("JobDescription", "")
        })

    return experience


def restructure_resume(input_json: Dict[str, Any]) -> Dict[str, Any]:
    rpd = input_json.get("ResumeParserData", {})

    return {
        "personal_info": extract_personal_info(rpd),
        "education": extract_education(rpd),
        "skills": extract_skills(rpd),
        "work_experience": extract_experience(rpd)
    }

In [10]:
def process_resume_directory(input_dir, output_dir):

    os.makedirs(output_dir, exist_ok=True)
    json_files = [f for f in os.listdir(input_dir) if f.lower().endswith(".json")]
    print(f"Found {len(json_files)} JSON files to process.")

    for fname in json_files:
        input_path = os.path.join(input_dir, fname)
        output_path = os.path.join(output_dir, fname)

        try:
            with open(input_path, "r", encoding="utf-8") as f:
                raw = json.load(f)

            cleaned = restructure_resume(raw)

            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(cleaned, f, indent=2, ensure_ascii=False)

            print(f"Processed: {fname}")

        except Exception as e:
            print(f"Failed: {fname} — {e}")


In [ ]:
if __name__ == "__main__":
    input_dir = (
        "/Users/elizabethgranda/Documents/resume_parsing/factored_benchmark/resume-parsing-exploration/RChilli_analysis/rchilli_output"
    )

    output_dir = (
        "/Users/elizabethgranda/Documents/resume_parsing/factored_benchmark/resume-parsing-exploration/RChilli_analysis/structured_rchilli_output"
    )

    process_resume_directory(input_dir, output_dir)
